# Cinema 興行収入予測器

このノートブックでは、線形回帰モデルを使用して映画の興行収入を予測します。

## 概要

- **問題タイプ**: 回帰問題
- **モデル**: OLS (Ordinary Least Squares)
- **特徴量**: SNS1, SNS2, actor, original
- **目的変数**: sales (興行収入)

## 1. ライブラリの読み込み

In [ ]:
(require '[ml-tdd-project.ml.cinema-predictor :as cp])
(require '[tablecloth.api :as tc])

## 2. データの読み込みと確認

In [ ]:
;; データの読み込み
(def data (cp/load-data "resources/data/cinema.csv"))
(def X (first data))
(def y (second data))

(println "データセットサイズ:" (tc/row-count X) "行")
(println "特徴量:" (vec (tc/column-names X)))
(println "目的変数の範囲:" (apply min y) "~" (apply max y))

## 3. データの確認

In [ ]:
;; 最初の5行を表示
(println "\n最初の5行:")
(tc/head X 5)

In [ ]:
;; 基本統計量
(println "\n基本統計量:")
(doseq [col (tc/column-names X)]
  (let [col-data (tc/column X col)]
    (println (format "%s: 平均=%.2f, 最小=%.2f, 最大=%.2f"
                    col
                    (/ (reduce + col-data) (count col-data))
                    (apply min col-data)
                    (apply max col-data)))))

## 4. 訓練・テストデータの分割

In [ ]:
;; データの分割 (80% train, 20% test)
(def n (tc/row-count X))
(def shuffled-indices (shuffle (vec (range n))))
(def train-size (int (* 0.8 n)))
(def train-indices (vec (take train-size shuffled-indices)))
(def test-indices (vec (drop train-size shuffled-indices)))

(def X-train (tc/select-rows X train-indices))
(def X-test (tc/select-rows X test-indices))
(def y-train (vec (map #(nth y %) train-indices)))
(def y-test (vec (map #(nth y %) test-indices)))

(println "訓練データ:" (count y-train) "サンプル")
(println "テストデータ:" (count y-test) "サンプル")

## 5. モデルの訓練

In [ ]:
;; 予測器の作成と訓練
(def predictor (cp/create-predictor))
(def trained-predictor (cp/train predictor X-train y-train))

(println "モデルの訓練が完了しました")
(println "モデルタイプ:" (type (:model trained-predictor)))

## 6. 予測と評価

In [ ]:
;; 予測
(def y-train-pred (cp/predict trained-predictor X-train))
(def y-test-pred (cp/predict trained-predictor X-test))

;; 評価
(def train-metrics (cp/evaluate y-train y-train-pred))
(def test-metrics (cp/evaluate y-test y-test-pred))

(println "\n【訓練データ】")
(println (format "  RMSE: %.2f" (:rmse train-metrics)))
(println (format "  R²:   %.4f" (:r2 train-metrics)))

(println "\n【テストデータ】")
(println (format "  RMSE: %.2f" (:rmse test-metrics)))
(println (format "  R²:   %.4f" (:r2 test-metrics)))

## 7. サンプル予測の表示

In [ ]:
(println "\nサンプル予測 (テストデータから10件):")
(println "実際値 | 予測値 | 誤差    | 誤差率")
(println "-------|--------|---------|--------")
(doseq [i (take 10 (range (count y-test)))]
  (let [actual (nth y-test i)
        predicted (nth y-test-pred i)
        error (- actual predicted)
        error-rate (* 100 (/ (Math/abs error) actual))]
    (println (format "%7.1f | %6.1f | %7.1f | %5.2f%%"
                    actual predicted error error-rate))))

## 8. 予測精度の分析

In [ ]:
;; 残差の計算
(def residuals (map - y-test y-test-pred))

(println "\n残差の統計:")
(println (format "  平均: %.2f" (/ (reduce + residuals) (count residuals))))
(println (format "  標準偏差: %.2f" 
                (Math/sqrt (/ (reduce + (map #(* % %) residuals)) 
                             (count residuals)))))
(println (format "  最小: %.2f" (apply min residuals)))
(println (format "  最大: %.2f" (apply max residuals)))

## 9. まとめ

このノートブックでは、以下を実装しました：

1. **データの読み込みと前処理**
   - 欠損値の補完
   - 外れ値の除外

2. **線形回帰モデルの訓練**
   - OLS (Ordinary Least Squares) を使用
   - 訓練データとテストデータで評価

3. **評価メトリクス**
   - RMSE: 予測誤差の平均
   - R²: 決定係数（モデルの説明力）

### 結果の解釈

- R² が 0.9 以上であれば、モデルは非常に良好な性能を示しています
- RMSE が小さいほど、予測精度が高いことを示します
- 残差の平均が 0 に近く、分布が正規分布に従っていれば、モデルは適切です